**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Array Processing & Beamforming

Filtering in **space**: with several microphones/antennas, you can point a 'listening beam' at a direction, null an interferer, and locate sources — all with the same linear algebra as temporal filtering. Three sessions from array geometry to MUSIC.

## 1. Pre-requisites

- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) (complex exponentials, DFT).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3–S4 (eigen/subspaces) for Session 3.
- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) S3 for MVDR.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# Uniform linear array (ULA): M sensors, half-wavelength spacing
M = 8
def steering(theta_deg):
    """Array response to a far-field narrowband source at angle theta (broadside = 0°)."""
    theta = np.deg2rad(theta_deg)
    return np.exp(1j * np.pi * np.arange(M) * np.sin(theta))   # d = λ/2

---
### 🕐 Session 1 of 3 — *The Array Manifold* (~35 min)
**Goal:** understand why direction becomes a phase pattern across sensors.
**Builds on:** [DSP Foundations](./Foundations_of_Signal_Processing_1.ipynb) S4. &nbsp; **Feeds into:** Session 2 (beamforming).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Array Manifold</b></summary>

**Timing (~35 min).** 10 min geometry: why delay becomes phase · 10 min the dictionary between time and space · 8 min the demo · 7 min spatial aliasing.

**Board first — draw the wavefront hitting the array at an angle.** A plane wave arriving at $\theta$ reaches sensor $m$ a little later than sensor $m-1$, by a path difference of $d\sin\theta$. For a *narrowband* signal, that delay is equivalent to a phase shift. Across a uniform line the phases therefore advance linearly, and the whole session follows from that one picture. Draw it before writing $e^{j\pi m \sin\theta}$, or the exponential looks arbitrary.

**The dictionary is the session's real content — put it on the board and leave it up.** Sensors ↔ samples. Aperture ↔ record length. Beamwidth ↔ frequency resolution. Element spacing ↔ sampling interval. Spatial aliasing ↔ temporal aliasing. This room already knows every right-hand column from [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb), so nothing in the left column is genuinely new — it is the same mathematics on a different axis. Students who build this table stop treating array processing as a separate subject.

**Ask the room.** "Why half-wavelength spacing? Why not put the sensors further apart for a bigger aperture?" Because $\sin\theta$ ranges over $[-1, 1]$, so spacing beyond $\lambda/2$ makes the spatial frequency exceed the spatial Nyquist limit — two different directions produce identical phase patterns, and the array cannot tell them apart. Those are **grating lobes**, and they are literally aliasing in space. This is the most satisfying transfer in the workshop; let them derive it from the sampling theorem they already own.

**Then the follow-up that sharpens it.** "So a bigger aperture is better?" Yes, but you buy it with *more sensors*, not wider spacing — resolution scales with total aperture while spacing is pinned at $\lambda/2$. That tension is why radio astronomy builds interferometers with sparse arrays and clever processing rather than simply spreading elements out.

**Point at what the demo actually shows.** The stem plots are sampled sinusoids whose frequency increases with $\sin\theta$ — and note it is $\sin\theta$, not $\theta$. Near broadside ($\theta = 0$) a few degrees of change barely alters the pattern, while near endfire the same few degrees change it a lot. That nonlinearity means angular resolution is *not uniform*: arrays resolve best at broadside and worst at endfire, which matters when siting a real array.

**Compute the numbers you will need later.** With $M = 8$ and $d = \lambda/2$ the aperture is 4 wavelengths, so the classical beamwidth is roughly $1/4$ radian, about **14°**. Have the room work that out now — Session 3 resolves two sources 8° apart, and the result only lands if they already know 8° is comfortably inside the classical limit.
</details>

## 2. Direction Is a Spatial Frequency

💡 **Intuition.** A far-field wavefront hits each sensor at a slightly different time; for a narrowband signal, delay ≈ phase shift. Across a uniform line of sensors the phases advance *linearly* — direction $\theta$ shows up as a **spatial sinusoid** with frequency $\propto \sin\theta$. Everything you know about temporal frequencies transfers verbatim: sensors ↔ samples, aperture ↔ record length, beamwidth ↔ resolution, and spacing $> \lambda/2$ ⇒ *spatial aliasing* (grating lobes) — Nyquist in space.

In [2]:
# The steering vector IS a sampled sinusoid — see it
fig, axes = plt.subplots(1, 3, figsize=(10, 2.4), sharey=True)
for ax, th in zip(axes, [0, 20, 60]):
    a = steering(th)
    ax.stem(np.arange(M), a.real, basefmt=" ")
    ax.set_title(f"θ = {th}°: spatial freq ∝ sin θ")
    ax.set_xlabel("sensor")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2028716/4231524107.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three steering vectors, plotted across the eight sensors, and each is a **sampled sinusoid**. At $\theta = 0°$ the wavefront arrives at every sensor simultaneously, so the phase is constant — spatial DC. At 20° the phases advance slowly across the array; at 60° they advance rapidly. Direction has become frequency, and the sensor index is playing the role of time.

That is the entire conceptual move of the workshop. Everything this room learned in [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) transfers with a change of variable names: sensors are samples, the array aperture is the record length, beamwidth is frequency resolution, and element spacing is the sampling interval. Beamforming will turn out to be a DFT evaluated at one spatial frequency, and direction finding a spectral estimation problem. Nothing new is being invented — a familiar toolkit is being pointed at a different axis.

**Two consequences worth reading off these plots.**

The spatial frequency is proportional to $\sin\theta$, not to $\theta$. Compare the 0° and 20° panels against the 20° and 60° panels: the same 20-degree step changes the pattern much less near broadside than near endfire... and in fact rather *more* between 0° and 20° than between 40° and 60°, since $\sin$ flattens as it approaches 1. Either way the mapping is nonlinear, so angular resolution is not uniform across the field of view — arrays see most sharply at broadside and worst at endfire, which is a real siting consideration.

And because $\sin\theta$ is bounded by 1, the spatial frequency is bounded too. Half-wavelength spacing places that maximum exactly at the spatial Nyquist limit. Space the elements further apart and distinct directions start producing identical phase patterns — **grating lobes**, which are aliasing in space, indistinguishable from the real thing and unfixable after the fact. The $\lambda/2$ in the code is the sampling theorem, enforced.

Note finally that only the real part is plotted. The steering vector is complex, and the imaginary part carries the other half of the phase information — which is why a real-valued array cannot distinguish $+\theta$ from $-\theta$ without additional structure.

---
### 🕐 Session 2 of 3 — *Delay-and-Sum & MVDR* (~40 min)
**Goal:** steer beams; then let the data place nulls on interferers automatically.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (subspace methods).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Delay-and-Sum & MVDR</b></summary>

**Timing (~40 min).** 8 min delay-and-sum as a spatial matched filter · 12 min the MVDR constrained problem · 12 min the demo and its null · 8 min the practical caveats.

**Delay-and-sum in one move.** Phase-align the sensors toward $\theta_0$ and add. Signals from $\theta_0$ stack coherently and gain $M$ in amplitude; everything else partially cancels. Point out that $w = a(\theta_0)/M$ is literally a *matched filter in space* — correlate against the expected pattern — and that scanning it across all angles is a DFT of the sensor data. Students who have done [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) can then predict the beam pattern's shape (a sinc-like main lobe with sidelobes) without computing anything, because it is the transform of a rectangular window.

**Then name its blind spot.** A matched filter is optimal in *white* noise. Interference is not white — it comes from a specific direction with real structure — and delay-and-sum has no mechanism to exploit that. Its pattern is fixed by geometry alone, chosen before any data arrives. Ask what a smarter beamformer would need: it would have to *look at the data*.

**MVDR as a two-line optimisation.** Minimise output power $w^H R w$ subject to $w^H a(\theta_0) = 1$. Say what each half does: the constraint protects the desired direction (unit gain, always), and the objective then suppresses everything it can. Since the only way to reduce total output power without touching $\theta_0$ is to reject whatever else is loud, nulls appear *automatically* wherever the covariance reports energy. Nobody tells the algorithm where the interferer is — it is inferred from $R$. That is the moment of the session, and it is a [Lagrange multiplier problem](../Intro_Math/Optimization/Optimization.ipynb) the room can solve in three lines if you have time.

**Ask the room before running.** "The interferer is 3× the amplitude of the desired signal — about 9.5 dB stronger. What SINR do you expect from each beamformer?" Then reveal 7.0 dB and 18.4 dB. The 11.4 dB gap is entirely the null, and it costs nothing but knowing $R$.

**Do not oversell MVDR — it is famously brittle, and this is the practically important part.** Three failure modes worth naming: (1) *steering vector mismatch* — if the assumed $\theta_0$ is even slightly wrong, the constraint protects the wrong direction and MVDR happily nulls the signal you wanted, performing far worse than delay-and-sum; (2) *finite snapshots* — $R$ is estimated from data, and with fewer snapshots than sensors it is singular, which is why real systems use diagonal loading, $R + \epsilon I$; (3) *coherent sources* — multipath copies of the same signal break the covariance structure the method relies on. Here we had 4000 clean snapshots, exact steering vectors, and independent sources: the best case in every respect. Say so.

**If the demo misbehaves.** `np.linalg.solve(R, a0)` will fail or produce nonsense if `N` is cut below `M`. That is a useful accident — it is exactly the singular-covariance failure above, and the fix (diagonal loading) is a one-line demonstration of why production beamformers all carry it.
</details>

## 3. Beamforming

💡 **Intuition.** **Delay-and-sum**: phase-align the sensors toward $\theta_0$ and add — signals from $\theta_0$ stack coherently ($M\times$ amplitude), others partially cancel. It's a matched filter in space, and like all matched filters it's optimal in white noise but naive about *structured* interference. **MVDR (Capon)** fixes that: minimize output power subject to unit gain at $\theta_0$ — $\mathbf{w} = \frac{R^{-1} \mathbf{a}}{\mathbf{a}^H R^{-1} \mathbf{a}}$ — a [Lagrange problem](../Intro_Math/Optimization/Optimization.ipynb) whose solution *automatically digs nulls* wherever the covariance says energy is coming from.

In [3]:
# Scene: desired source at 0°, LOUD interferer at 40°, noise
theta_s, theta_i = 0, 40
N = 4000
s = rng.standard_normal(N)                      # desired
i_sig = 3.0 * rng.standard_normal(N)            # interferer, 3x amplitude
X = (np.outer(steering(theta_s), s) + np.outer(steering(theta_i), i_sig)
     + 0.3 * (rng.standard_normal((M, N)) + 1j * rng.standard_normal((M, N))) / np.sqrt(2))

R = X @ X.conj().T / N
a0 = steering(theta_s)

w_das = a0 / M
w_mvdr = np.linalg.solve(R, a0); w_mvdr /= (a0.conj() @ w_mvdr)

angles = np.linspace(-90, 90, 721)
def pattern(w):
    return np.array([np.abs(w.conj() @ steering(t))**2 for t in angles])

plt.figure(figsize=(9, 3))
plt.plot(angles, 10*np.log10(pattern(w_das)), label="delay-and-sum")
plt.plot(angles, 10*np.log10(pattern(w_mvdr)), label="MVDR")
for th, name in [(theta_s, "source"), (theta_i, "interferer")]:
    plt.axvline(th, color="k", linestyle=":", linewidth=0.8)
plt.ylim(-60, 5); plt.legend(); plt.xlabel("angle [deg]"); plt.ylabel("gain [dB]")
plt.title("MVDR digs a null exactly at the 40° interferer — nobody told it to")
plt.tight_layout(); plt.show()

for name, w in [("delay-and-sum", w_das), ("MVDR", w_mvdr)]:
    y = w.conj() @ X
    sinr = np.var(s) * np.abs(w.conj() @ a0)**2 / np.var(y - (w.conj() @ a0) * s)
    print(f"{name:14s} output SINR ≈ {10*np.log10(sinr.real):5.1f} dB")

delay-and-sum  output SINR ≈   7.0 dB
MVDR           output SINR ≈  18.4 dB


/tmp/ipykernel_2028716/2357919198.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Output SINR of **7.0 dB** for delay-and-sum against **18.4 dB** for MVDR — an 11.4 dB improvement from the same eight sensors and the same data. The interferer at 40° is 3× the desired amplitude, about 9.5 dB stronger in power, and the two beamformers deal with it completely differently.

**Read the patterns, because they explain the numbers.** Delay-and-sum has the beam shape you would predict from [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) alone: a main lobe at 0° and a fixed sidelobe structure that is the transform of a rectangular window. Crucially that pattern was determined *before any data arrived* — it depends only on geometry. Wherever the interferer happens to land, it is attenuated by whatever the sidelobe level is there and no more, which here leaves it dominating the output.

MVDR's pattern keeps unit gain at 0° and drives a deep, narrow null down at exactly 40°. Nothing in the code mentions 40°. The interferer's direction is never estimated, never passed as a parameter, never searched for. It emerges from $R$ alone: the constraint $w^H a(\theta_0) = 1$ pins the desired direction, and minimising $w^H R w$ then suppresses everything else it can — and the only large "everything else" is at 40°. Nulls are what optimality *does* when there is structured interference to reject.

**Why the SINR gap is smaller than the null is deep.** The null is 40+ dB, but SINR only improves by 11.4 dB, because once the interferer is removed the output is limited by the remaining white noise — which MVDR cannot beat, since suppressing spatially white noise is exactly what delay-and-sum already does optimally. MVDR wins on the *structured* component and ties on the unstructured one.

**Now the honest framing, because MVDR is famously brittle.** This demo is the best case in every respect: 4000 snapshots to estimate an $8\times 8$ covariance, exact steering vectors, and sources that are independent. Change any of these and it degrades sharply.

- **Steering vector mismatch.** The constraint protects the direction you *claim* is the signal. Get $\theta_0$ slightly wrong — through calibration error, element position error, or an unmodelled array response — and MVDR treats the real signal as interference and nulls it. It can then perform *worse* than delay-and-sum, which has no such failure mode. This is the classic "signal cancellation" problem.
- **Finite snapshots.** $R$ here is estimated, not known. With fewer snapshots than sensors it is singular and `solve` fails; even with a few times $M$ it is poorly conditioned. Production systems add diagonal loading, $R + \epsilon I$, which trades a little null depth for robustness.
- **Coherent sources.** Multipath produces correlated copies of the same signal, which breaks the covariance structure the method assumes; spatial smoothing is the standard repair.

The general shape is worth carrying away: adaptive methods buy large gains by exploiting structure in the data, and pay for it with sensitivity to whether that structure is really what you assumed. Session 3's MUSIC makes the same bargain, more aggressively.

---
### 🕐 Session 3 of 3 — *Subspace Methods: MUSIC* (~40 min)
**Goal:** use covariance eigenstructure to localize sources beyond the beamwidth limit.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3–S4.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Subspace Methods — MUSIC</b></summary>

**Timing (~40 min).** 10 min the subspace decomposition · 10 min why orthogonality gives resolution · 12 min the demo · 8 min what MUSIC needs to work.

**Set up the impossibility first.** Remind the room of Session 1's arithmetic: an 8-element array with $\lambda/2$ spacing has a 4-wavelength aperture, so the classical beamwidth is about **14°**. Now announce the two sources are 8° apart — comfortably *inside* one beamwidth. Ask whether they can be resolved. By classical reasoning, no: any beam pointed at one substantially admits the other, which is exactly what the left panel of the demo shows. Then say MUSIC will separate them anyway. That framing is what makes the session land; without it, two peaks on a plot look unremarkable.

**Board first — the subspace split.** With $K$ sources and $M$ sensors, the covariance $R$ has $K$ large eigenvalues and $M-K$ small ones. The eigenvectors of the large ones span the **signal subspace**, and the true steering vectors live inside it. The rest span the **noise subspace**, which is orthogonal to every true steering vector. Draw it as two perpendicular blocks; the geometry is the whole method.

**Then the trick.** Rather than asking "how much power comes from $\theta$?" — the classical question, limited by beamwidth — MUSIC asks "how *orthogonal* is $a(\theta)$ to the noise subspace?" At a true direction the projection is essentially zero, so its reciprocal explodes. The pseudo-spectrum's peaks are therefore not power measurements at all, and this is the key conceptual point: **the y-axis is not power, and the peak heights mean nothing physical.** Students routinely read them as source strengths. They are reciprocals of a projection residual, and a weak source with a clean subspace can peak higher than a strong one.

**Why the resolution limit is beaten.** Beamwidth limits how finely you can *resolve power*. Orthogonality is a sharper test — it is a yes/no geometric condition, and it stays sharp as long as the eigen-decomposition cleanly separates signal from noise. That separation is what noise and finite snapshots erode, which is why MUSIC's resolution degrades gracefully with SNR rather than being fixed by geometry.

**The load-bearing assumption to state plainly.** `K = 2` is *given* in the code. MUSIC must be told how many sources there are, and getting $K$ wrong breaks it: too small leaves signal energy in the "noise" subspace and buries real peaks; too large eats the noise subspace and manufactures spurious ones. In practice $K$ is estimated first — by AIC or MDL on the eigenvalue spectrum — and that estimation is its own hard problem. Ask the room to look at the eigenvalues and judge where the gap is; on clean data it is obvious, and it is a good exercise to find where it stops being obvious.

**Two further caveats worth naming.** Coherent sources (multipath) make the signal covariance rank-deficient and MUSIC fails outright without spatial smoothing. And the sharpness is a property of the *pseudo-spectrum*, not evidence of precision — a razor peak in the wrong place is still wrong, and MUSIC's variance at low SNR is genuinely large despite how confident the plot looks.

**If the demo misbehaves.** The peak-picking line is dense and fragile — it differentiates the sign of the differences to find local maxima. If it returns odd values, that is the peak finder, not MUSIC; the plot is the reliable evidence. Reducing `N` below a few hundred, or raising the noise, makes the two peaks merge, which is a good way to find the method's actual limit live.
</details>

## 4. MUSIC

💡 **Intuition.** With $K$ sources, the covariance's top-$K$ eigenvectors span the **signal subspace** (where steering vectors of true directions live); the remaining eigenvectors span the orthogonal **noise subspace**. MUSIC scans directions and scores each by *how orthogonal its steering vector is to the noise subspace* — true directions produce near-zero projections and towering pseudo-spectrum peaks. This is [Linear Algebra S3](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)'s eigen-story paying rent: resolution beyond the classical beamwidth.

In [4]:
# Two sources only 8° apart — closer than the array's beamwidth
th1, th2, K = -3, 5, 2
S2 = np.stack([rng.standard_normal(N), rng.standard_normal(N)])
A = np.stack([steering(th1), steering(th2)], axis=1)
X2 = A @ S2 + 0.5 * (rng.standard_normal((M, N)) + 1j*rng.standard_normal((M, N))) / np.sqrt(2)
R2 = X2 @ X2.conj().T / N

evals, evecs = np.linalg.eigh(R2)
En = evecs[:, :M-K]                                  # noise subspace (small eigenvalues)

das_spec = np.array([np.abs(steering(t).conj() @ (R2 @ steering(t))).real for t in angles])
music = np.array([1 / np.linalg.norm(En.conj().T @ steering(t))**2 for t in angles])

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.8))
axes[0].plot(angles, 10*np.log10(das_spec / das_spec.max()))
axes[0].set_title("classical scan: one blurred bump"); axes[0].set_xlim(-40, 40)
axes[1].plot(angles, 10*np.log10(music / music.max()))
axes[1].set_title("MUSIC: two razor peaks at −3° and 5°"); axes[1].set_xlim(-40, 40)
for ax in axes:
    for th in (th1, th2): ax.axvline(th, color="k", linestyle=":", linewidth=0.8)
    ax.set_xlabel("angle [deg]")
plt.tight_layout(); plt.show()

peaks = angles[np.argsort(music)[-2:]] if False else angles[(np.diff(np.sign(np.diff(music))) < 0).nonzero()[0] + 1]
top2 = peaks[np.argsort(music[np.searchsorted(angles, peaks)])[-2:]]
print("MUSIC peak estimates:", np.sort(np.round(top2, 1)), " (truth: [-3, 5])")

MUSIC peak estimates: [-3.  5.]  (truth: [-3, 5])


/tmp/ipykernel_2028716/3945217473.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** MUSIC recovered `[-3. 5.]` against a truth of `[-3, 5]` — both directions, exactly, from two sources separated by **8°** on an array whose classical beamwidth is about **14°**. The left panel shows what classical processing can do with the same data: a single blurred bump that gives no hint there are two sources at all.

That comparison is the result. The right panel is not a better-tuned version of the left one; the two panels answer different questions. The classical scan measures *power arriving from* each direction, and power measurement is limited by beamwidth — a 4-wavelength aperture cannot resolve finer, and no amount of processing changes the geometry. MUSIC instead measures *orthogonality to the noise subspace*, which is a geometric yes/no condition with no beamwidth attached.

**The mechanism, concretely.** With 2 sources and 8 sensors, $R$ has 2 large eigenvalues and 6 small ones. The 6 small eigenvectors span a noise subspace that is orthogonal to every true steering vector. Scanning $\theta$ and computing $1/\|E_n^H a(\theta)\|^2$ therefore produces a near-division-by-zero precisely at $-3°$ and $5°$, and nothing special anywhere else. The peaks are sharp because the projection collapses fast as $\theta$ approaches truth, not because the array suddenly acquired more aperture.

**Read the y-axis correctly — this is the most common misreading.** The pseudo-spectrum is *not* power. Its height is the reciprocal of a projection residual, so peak heights carry no physical meaning: a weak source with a clean subspace can peak higher than a strong one. MUSIC tells you *where* sources are, not how loud they are. If you need amplitudes, you estimate them separately once the directions are known.

**And the assumptions this rests on, which are substantial.** `K = 2` is *given* to the algorithm, not discovered — the line `En = evecs[:, :M-K]` needs to know how many sources exist. Get $K$ wrong and the method breaks in both directions: too small leaves signal energy inside the supposed noise subspace and buries the real peaks; too large consumes the noise subspace and invents spurious ones. Real systems estimate $K$ first from the eigenvalue spectrum (AIC, MDL), and that is its own difficult problem.

Beyond that: the sources here are *independent*, so the signal covariance has full rank $K$. Coherent sources — multipath, the normal case in a room or an urban channel — make it rank-deficient and MUSIC fails outright, needing spatial smoothing to repair. And we had 4000 snapshots at good SNR; the sharpness of these peaks reflects a clean eigen-decomposition, not intrinsic precision. Lower the SNR or shorten the record and the peaks broaden, then merge. A razor-sharp peak in the wrong place is still wrong, and this plot's confidence is not itself evidence.

**The bargain, stated once.** Classical beamforming assumes almost nothing and is limited by physics. Subspace methods assume a specific model — known $K$, independent sources, accurate steering vectors — and in exchange beat the physical limit. That is the same trade MVDR made in Session 2, pushed further: more structure assumed, more performance when the assumption holds, more catastrophic failure when it does not.

## 5. Conclusion

Direction = spatial frequency; delay-and-sum = spatial matched filter; MVDR = constrained optimization that nulls interference by itself; MUSIC = eigen-subspace geometry beating the beamwidth. Space is just another axis to filter.

---
## Where next

- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) — the covariance machinery underneath.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — arrays of real antennas.
- [Audio & Speech DSP](./Audio_Speech_DSP.ipynb) — microphone arrays in your smart speaker run exactly this.